In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_ollama import ChatOllama


# =========================================================
# 1. CUSTOM EXCEPTION
# =========================================================

class BranchNotFoundError(Exception):
    pass


# =========================================================
# 2. FAKE DATABASE
# =========================================================

BRANCH_SCORES = {
    "101": 87.5,
    "102": 91.2,
}


# =========================================================
# 3. TOOL
# =========================================================

@tool
def get_branch_score(branch_code: str) -> str:
    """
    Get the performance score of a bank branch.
    """

    if branch_code not in BRANCH_SCORES:

        raise BranchNotFoundError(
            f"Branch {branch_code} was not found."
        )

    score = BRANCH_SCORES[branch_code]

    return (
        f"Branch {branch_code} "
        f"score = {score}"
    )


# =========================================================
# 4. ERROR HANDLING MIDDLEWARE
# =========================================================

@wrap_tool_call
def error_handler(request, handler):

    try:

        return handler(request)

    except BranchNotFoundError:

        return ToolMessage(
            content=(
                "The requested branch does not exist."
            ),
            tool_call_id=request.tool_call["id"],
        )

    except Exception as exc:

        print(
            "Unexpected tool error:",
            type(exc).__name__,
            str(exc),
        )

        return ToolMessage(
            content=(
                "The tool could not be executed "
                "because of an unexpected error."
            ),
            tool_call_id=request.tool_call["id"],
        )


# =========================================================
# 5. MODEL
# =========================================================

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)


# =========================================================
# 6. AGENT
# =========================================================

agent = create_agent(
    model=model,

    tools=[
        get_branch_score,
    ],

    middleware=[
        error_handler,
    ],
)


# =========================================================
# 7. REQUEST
# =========================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "امتیاز شعبه 999 را به من بگو."
                ),
            }
        ]
    }
)


# =========================================================
# 8. RESULT
# =========================================================

print(
    result["messages"][-1].content
)

Error
 ├── Business Error
 ├── Authorization Error
 ├── Validation Error
 ├── Transient Error
 └── Unexpected Error